In [27]:
import os
import json
from groq import Groq
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import List
import re
import json

In [28]:
load_dotenv()

llm = Groq(api_key=os.getenv("GROQ_API_KEY"))

MODEL_NAME = "llama-3.1-8b-instant"


In [29]:
class MCQ(BaseModel):
    question: str
    options: List[str]
    answer: str


class GeneratorOutput(BaseModel):
    explanation: str
    mcqs: List[MCQ]


class ReviewerOutput(BaseModel):
    status: str  # "pass" or "fail"
    feedback: List[str]


In [30]:
GENERATOR_PROMPT = """
You are an educational content generator.

Grade: {grade}
Topic: {topic}

{feedback_block}

Rules:
- Use language suitable for Grade {grade}
- Be conceptually correct
- Return ONLY valid JSON in this structure:
{{
  "explanation": "...",
  "mcqs": [
    {{
      "question": "...",
      "options": ["A", "B", "C", "D"],
      "answer": "A"
    }}
  ]
}}
"""

REVIEWER_PROMPT = """
You are an educational reviewer.

Evaluate the following content for:
- Age appropriateness
- Conceptual correctness
- Clarity

Content:
{content}

Return ONLY valid JSON:
{{
  "status": "pass" or "fail",
  "feedback": ["..."]
}}
"""


In [ ]:
def generator_agent(grade, topic, feedback=None):
    feedback_block = ""
    if feedback:
        feedback_block = f"Reviewer feedback to fix:\n{feedback}"
    prompt = GENERATOR_PROMPT.format(
        grade=grade,
        topic=topic,
        feedback_block=feedback_block
    )

    response = llm.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_format={"type": "json_object"}  
    )
    parsed = response.choices[0].message.content
    return GeneratorOutput(**json.loads(parsed))


In [ ]:
def reviewer_agent(generator_output):
    prompt = REVIEWER_PROMPT.format(
        content=generator_output.json()
    )

    response = llm.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_format={"type": "json_object"}    )

    parsed = response.choices[0].message.content
    return ReviewerOutput(**json.loads(parsed))


In [34]:
def run_pipeline(grade, topic):
    gen_output = generator_agent(grade, topic)
    review_output = reviewer_agent(gen_output)

    refined_output = None

    if review_output.status == "fail":
        gen_output_refined = generator_agent(
            grade, topic, feedback=review_output.feedback
        )
        refined_output = gen_output_refined

    return gen_output, review_output, refined_output


In [37]:
gen, review, refined = run_pipeline(
    grade=12,
    topic="Calculus"
)

print("GENERATOR OUTPUT:\n", gen)
print("\nREVIEWER OUTPUT:\n", review)

if refined:
    print("\nREFINED OUTPUT:\n", refined)


C:\Users\Jayant\AppData\Local\Temp\ipykernel_11448\2613089889.py:3: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  content=generator_output.json()


GENERATOR OUTPUT:
 explanation='Calculus is a branch of mathematics that deals with the study of continuous change, particularly in the context of functions and limits. It consists of two main branches: Differential Calculus and Integral Calculus. Differential Calculus focuses on the study of rates of change and slopes of curves, while Integral Calculus deals with the study of accumulation of quantities.' mcqs=[MCQ(question='What is the primary focus of Differential Calculus?', options=['Rates of change and slopes of curves', 'Accumulation of quantities', 'Limits and continuity', 'Optimization problems'], answer='A'), MCQ(question='What is the fundamental theorem of Calculus?', options=['The derivative of a function is equal to the slope of the tangent line', 'The integral of a function is equal to the accumulation of the area under the curve', "The limit of a function as x approaches a certain value is equal to the function's value at that point", "The derivative of a function is equa